# Notebook to output STM from a .zarr stack


In [30]:
# Standard packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Parallel processing packages
import dask
import xarray as xr
import zarr
import sarxarray

# For plotting 
import contextily as cx
import pyproj

# Optimization packages
import scipy.io
# from scipy.optimize import least_squares, curve_fit
import scipy.stats as st
# from scipy.stats import norm,rayleigh,rice,vonmises

# Import required toolboxes
import sys
sys.path.append('/Users/wietskebrouwer/surfdrive/Documents/PhD/07 Projecten/13 InSAR-AAA/toolbox')
# from blue_testing import *
from point_quality import *
from general import *
# from deformation_models import *
from arc_estimation_toolbox import *
# from implicit_unwrapping import *
from stack_nmad_sel import *


# INPUT parameters

3 OPTIONS:
- Not do it
- Start time and end time
- specified start and end time

In [32]:
# Data location
slc_path = '/Users/wietskebrouwer/surfdrive/Documents/PhD/07 Projecten/08 Paramatric approach/data/computed_slcs_s1_dsc_t037.zarr'
mother_path = '/Users/wietskebrouwer/surfdrive/Documents/PhD/07 Projecten/08 Paramatric approach/data/mother_slc_s1_dsc_t037.zarr'
h2ph_path = '/Users/wietskebrouwer/surfdrive/Documents/PhD/07 Projecten/08 Paramatric approach/data/h2ph_values_s1_dsc_t037.zarr'
weather_data_path = '/Users/wietskebrouwer/surfdrive/Documents/PhD/07 Projecten/08 Paramatric approach/data/weather_data_schiphol.txt'

# Crop in space 
lat_low = 52.355
lat_high = 52.391
lon_low = 4.86
lon_high = 4.93

# Crop in time
last_date = datetime(2020, 9, 1, 0, 0)
last_date = datetime(2025, 9, 1, 0, 0)
first_date = datetime(2014, 1, 1, 0, 0)

start_data_ps_selection = 0 ### ADD
initialization_length = 0 ### ADD


# PS selection method
nmad_max = 0.25
nad_max = 0.3
ps_selection_method = "nmad"
chunks_ps_selection = 1000
nr_points_selection = 60


# Input variables for the outlier detection
do_outlier_detection = 1
drop_outliers = 1
window_size_outliers = 15
outlier_detection_db = 1


# Input variables for the breakpoint detection
search_method = 'pelt'
cost_function = 'l2'
db_segmentation = 0
min_obs_segment = 27


# Compute temporal differences
mother_epoch_sd = '20190807'



# Load SLC data and create one stack

In [ ]:
slcs = xr.open_zarr(slc_path)
mother_slc = xr.open_zarr(mother_path)

# Note that there is a naming 'problem' between the latitude and longitude. They are switched
"""@SIMON
The two lines of code below are not needed in the new version
"""
slcs = slcs.rename({'lat': 'lon', 'lon': 'lat'})
mother_slc = mother_slc.rename({'lat': 'lon', 'lon': 'lat'})


"""@Simon
The block below is not needed in the new version """

################################################################################################
######################## ADD THE MOTHER IMAGE TO THE STACK #####################################
################################################################################################

# Extract the mother date 
mother_date = mother_slc.time.values[0]
mother_date

# Look where the mother date should be added in the slc datastack
insert_index = (np.where((slcs.time < mother_date)))[0][-1] + 1
mother_index = insert_index
print (f'The mother date is {mother_date}, and the index is {mother_index}')


# Add the mother slc at the right location in the stack
slcs_all = xr.concat([slcs.isel(time=slice(0, insert_index)), mother_slc, slcs.isel(time=slice(insert_index, None))], dim='time')


# Extract the complex data
complex_slcs = slcs_all.complex.compute()

# check whether everyting went fine with the mother slc
test_mother = complex_slcs[:,:,mother_index]
mother_check = np.angle(mother_slc.complex[:,::-1, 0]) - np.angle(test_mother[:,::-1])

# Give a warning if the mother is not added correctly to the slc stack
assert np.sum(mother_check) == 0, f'Somehting went wrong in adding the mother to the slc stack!'


################################################################################################
##################### EXTRACT DATES AND ADD AMPLITUDE AND PHASE DATA ###########################
################################################################################################

# add amplitude and phase data to the stack
slcs_all = slcs_all.slcstack._get_amplitude()
slcs_all = slcs_all.slcstack._get_phase()

# Extract dates
date_array = slcs_all.time.values

# Convert to datetime-objects
dates = [datetime.strptime(date_str, '%Y%m%d') for date_str in date_array]

# Convert back to other format
formatted_dates = [date_obj.strftime('%Y%m%d') for date_obj in dates]

# Show the MRM of the dataset
plt.figure(figsize = (10,7))
slcs_all.slcstack.mrm().plot(x='lon', y='lat', vmin = 0, vmax = 500, cmap = 'Greys_r')

# Load h2ph values

In [4]:
# Load h2ph values
h2ph_stack = xr.open_zarr(h2ph_path)
h2ph_stack = h2ph_stack.rename({'lat': 'lon', 'lon': 'lat'})

# Add the mother h2ph values to the slc stack
# Create a zero array with the same dimensions as h2ph_stack
# Since the h2ph values at the mother epoch are zero
zero_array = np.zeros((len(h2ph_stack.azimuth), len(h2ph_stack.range)))

# Create a new xarray dataset for the mother h2ph values
mother_h2ph = h2ph_stack.isel(time = 0)
mother_h2ph.complex 

mother_h2ph['complex'][:] = zero_array
mother_h2ph['time'] = [mother_date]

# Add the mother h2ph values to the stack
# Insert the zero array into the h2ph_stack
h2ph_stack = xr.concat([h2ph_stack.isel(time=slice(0, insert_index)), mother_h2ph, h2ph_stack.isel(time=slice(insert_index, None))], dim='time')

# Rename the datavariable
h2ph_stack = h2ph_stack.rename({'complex': 'h2ph_values'})



# Load weather data

In [ ]:
#Load weather data
weather_data = pd.read_csv(weather_data_path, sep=',', skiprows=51)

#Transform weather data to date time
weather_data['DateTime'] = weather_data.iloc[:,1].apply(lambda x: pd.to_datetime(str(x), format='%Y%m%d'))
date_time_weather= weather_data['DateTime']

first_insar_date = dates[0]

# Find the days with respect to t0 for the weather time series
days_weather = date_time_weather-first_insar_date

# Extract temperature time series at the days of acquisitions
# find dates equal to SAR dates

Temp = np.zeros(len(dates))
dates_test = np.zeros(len(dates))

for t in range(len(dates)):

    # The selection array only selects the data for the particular InSAR observation day. 
    selection = date_time_weather == dates[t]
    index = [i for i, x in enumerate(selection) if x]
    Temp[t] = weather_data.iloc[index,11]/10
    dates_test[t] = weather_data.iloc[index,1]


# Create subset in time and space of the stack

In [ ]:
# Mask in space
mask_subset = create_mask_xarray(lat_low, lat_high, lon_low, lon_high, slcs_all)
slc_subset = slcs_all.where(mask_subset, drop=True)
h2ph_subset = h2ph_stack.where(mask_subset, drop=True)

# Mask in time
mask_dates = [(date < last_date) and (date > first_date) for date in dates]
selected_dates = [date for date, mask in zip(dates, mask_dates) if mask]
slc_subset = slc_subset.sel(time=mask_dates)
h2ph_subset = h2ph_subset.sel(time=mask_dates)
date_array = slc_subset.time.values


# Display the mrm of the subset of slc's (both in space and in time)
fig, ax = plt.subplots(figsize = (10,6))
slc_subset.slcstack.mrm().plot(x='lon', y='lat', vmin = 0, vmax = 1000, alpha = 0.6, cmap = 'Greys_r')
plt.title('Mean Amplitude map of the subselection')
# cx.add_basemap(ax, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 12)




In [ ]:
# Apply point selection based on NAD and NMAD for the subselection
"""
Normaly I select points with NAD 0.3 and NMAD 0.25. 
Now I want to select more points for a nicer analysis to also test it on points that are less good. 
"""
stm, mask_1d_nmad, index_nmad =  Stack(slc_subset).point_selection(nmad_max, ps_selection_method, chunks=chunks_ps_selection)

# Apply the mask on the h2ph values as well
stm_h2ph_nmad = h2ph_from_stack_to_stm(h2ph_subset, index_nmad, chunks_ps_selection)

# Add the STM of the h2ph value to the stm with points
stm = stm.merge(stm_h2ph_nmad)

# Change the names of some variables
stm = stm.rename({'complex': 'slc_complex', 'amplitude': 'slc_amplitude', 'phase': 'slc_phase'})


print (f'We selected {len(stm.space)} points with the nmad selection')
print ('')
print (f'We selected {len(stm_h2ph_nmad.space)} points with the nmad selection in h2ph stack')

In [ ]:
fig, ax = plt.subplots(figsize = (15,15))
plt.scatter(stm.lon.data, stm.lat.data, s=30, color = 'tab:red', label = 'NMAD')
plt.legend()
cx.add_basemap(ax, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 15)

In [ ]:
# Select the nr of points that you want to select

np.random.seed(42)

# Generate a random selection of the points in space
random_indices_space = np.random.choice(len(stm.coords['lon']), nr_points_selection, replace=False)

# select the random selection in the stm
stm = stm.isel(space=random_indices_space)

# Plot the random selection
fig, ax = plt.subplots(figsize = (15,15))
plt.scatter(stm.lon.data, stm.lat.data, s=30, color = 'tab:red', label = 'NMAD')
plt.legend()
cx.add_basemap(ax, crs='epsg:4326', source=cx.providers.OpenStreetMap.Mapnik, zoom = 14)


# Partitioning and quality estimation

In [20]:
############################################################################################
################### 4A. ADD TEMPERATURE AND DAYS AND YEARS TO STM ##########################
############################################################################################

# Add temperature data to the stm
dates = [date for date in dates if date < last_date]
stm['temperature'] = xr.DataArray(Temp[mask_dates], dims=('time'), coords={'time': stm.time})
stm['dates'] = xr.DataArray(selected_dates, dims=('time'), coords={'time': stm.time})

# Add nr of days and nr of years to STM
dates = selected_dates

# Calulate nr of days since start of the time series
days = [(date - dates[0]).days for date in dates]
days = np.asarray(days)
years = days/365

stm['years'] = xr.DataArray(years, dims=('time'), coords={'time': stm.time})
stm['days'] = xr.DataArray(days, dims=('time'), coords={'time': stm.time})

# Add RD-x and RD-y coordinates to the STM
# To estimate the coordinate of the point in RD 
# Define required projection
wgs84 = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True).transform
# Hier wordt EPSG:4326 gebruikt voor WGS84 en EPSG:28992 voor RD (Rijksdriehoekscoördinaten)

# # Conert Lat and Long to  RD-coördinates
rd_x, rd_y = wgs84(stm['lon'].values, stm['lat'].values)

# Add RD  coordinates to the dataset
stm['rd_x'] = xr.DataArray(rd_x, dims='space')
stm['rd_y'] = xr.DataArray(rd_y, dims='space')


In [ ]:
############################################################################################
############### 4B. DEFINE BREAKPOINTS PER PS AND COMPUTE NAD AND NMAD #####################
############################################################################################

"""
Here the breakpoints (starting index of a new partition) and NAD, NMAD, sigma_ampl, median (ampl) per partition are deined
Also we compute the NAD and NMAD for the entire time series
Done for a list of the PS points and the reference point

"""
print ('4B. Defining Breakpoints and quality NAD, NMAD per detected partition')

# Estimate the NAD and NMAD for the entire time series based on the ampltiude 

amplitude_vals = stm.slc_amplitude.values

nad_points = np.zeros(len(stm.space))
nmad_points = np.zeros(len(stm.space))

for i in range(len(stm.space)):
    mean_ampl, sigma_ampl, nad_points[i] = NAD(amplitude_vals[i,:])
    median_ampl, mad_ampl, nmad_points[i] = NMAD(amplitude_vals[i,:])

# Add values to the stm matrix
stm['nad_full'] = xr.DataArray(nad_points, dims=('space'), coords={'space': stm.space})
stm['nmad_full'] = xr.DataArray(nmad_points, dims=('space'), coords={'space': stm.space})

# Drop the points with nans in amplitude time series
indices_no_nan = np.where(~np.isnan(nmad_points))[0]
stm = stm.isel(space=indices_no_nan)

amplitude_vals = stm.slc_amplitude.values


# define breakpoints for all PS points and NAD, NMAD, sigma_ampl, and median (ampl) per partition
breakpoints_ps, nad_ps, sigma_A_ps, nmad_ps, mad_ps = define_bkps_output_results(amplitude_vals, db_segmentation, 
                                                      search_method, cost_function, dates, min_obs_segment, 
                                                      0)


# Define space quality arrays and add them to the stm
# The cells below compute the breakspoints and slc quality per epoch
# The code above saves it as lists where each list for each point has a sifferent size 
nad_stm, breakpoints_stm = compute_bkps_nmad_stm(len(stm.space), len(stm.time), breakpoints_ps, nad_ps)
nmad_stm, breakpoints_stm = compute_bkps_nmad_stm(len(stm.space), len(stm.time), breakpoints_ps, nmad_ps)

# Estimate the phase dispersion based on the nmad
slc_quality_stm = NMAD_to_sigma_phase(nmad_stm, 'mean_2_sigma')
slc_quality_nad_stm = NAD_to_sigma_phase(nad_stm, 'mean_2_sigma')
slc_quality_stm_mean_cloud = NMAD_to_sigma_phase(nmad_stm, 'mean')
slc_quality_nad_stm_mean_cloud = NAD_to_sigma_phase(nad_stm, 'mean')

In [23]:
## Add the nmad, nad, breakpoints, and quality stm to the stm
breakpoints_ps = np.array(breakpoints_ps, dtype=object)
stm['breakpoints'] = xr.DataArray(breakpoints_ps, dims=('space'), coords={'space': stm.space})
stm['slc_quality'] = xr.DataArray(slc_quality_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['slc_quality_nad'] = xr.DataArray(slc_quality_nad_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['slc_quality_mean_cloud'] = xr.DataArray(slc_quality_stm_mean_cloud, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['slc_quality_nad_mean_cloud'] = xr.DataArray(slc_quality_nad_stm_mean_cloud, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['breakpoints_stm'] = xr.DataArray(breakpoints_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['nmad_stm'] = xr.DataArray(nmad_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['nad_stm'] = xr.DataArray(nad_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})



In [ ]:
%%time
# Add the quality information as arrays to the stm

sigma_ampl_slc_stm = np.zeros((len(stm.space), len(stm.time)))
mean_ampl_slc_stm = np.zeros((len(stm.space), len(stm.time)))
mad_ampl_slc_stm = np.zeros((len(stm.space), len(stm.time)))
median_ampl_slc_stm = np.zeros((len(stm.space), len(stm.time)))

slc_ampl_np = stm.slc_amplitude.values

for i in range(len(stm.space)):
        # sigma_ampl_slc_stm[i,:], mean_ampl_slc_stm[i,:], mad_ampl_slc_stm[i,:], median_ampl_slc_stm[i,:] = define_complex_stochastics_partition_stm(stm.isel(space = i).breakpoints_stm, stm.isel(space = i).slc_amplitude, 'normal')
        sigma_ampl_slc_stm[i,:], mean_ampl_slc_stm[i,:], mad_ampl_slc_stm[i,:], median_ampl_slc_stm[i,:] = define_complex_stochastics_partition_stm(breakpoints_stm[i,:], slc_ampl_np[i,:], 'normal')
    
stm['sigma_ampl_slc_stm'] = xr.DataArray(sigma_ampl_slc_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['mean_ampl_slc_stm'] = xr.DataArray(mean_ampl_slc_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['mad_ampl_slc_stm'] = xr.DataArray(mad_ampl_slc_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['median_ampl_slc_stm'] = xr.DataArray(median_ampl_slc_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})


In [ ]:
##############################################################################
######################## 4C. DETECT THE OUTLIERS #############################
##############################################################################
"""
Outliers are detected for the reference point and all other PS points. Data is saved for all points

"""

if do_outlier_detection == 1:
    print ('4C. Doing the outlier detection')
    
    # Define the outliers for the points
    idx_outliers, nr_outliers = detect_outliers_amplitude(window_size_outliers, amplitude_vals, outlier_detection_db, 0)
    outliers_stm = compute_outlier_stm(len(stm.space), len(stm.time),idx_outliers)

    idx_outliers = np.array(idx_outliers, dtype=object)
    stm['idx_outliers'] = xr.DataArray(idx_outliers, dims=('space'), coords={'space': stm.space})
    stm['nr_outliers'] = xr.DataArray(nr_outliers, dims=('space'), coords={'space': stm.space})
    stm['outliers_stm'] = xr.DataArray(outliers_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
    


In [31]:
##############################################################################
################ 4D. COMPUTE SINGLE (TEMPORAL) DIFFERENCES ###################
##############################################################################

# Choose the epoch of the mother date
mother_idx = np.where(stm.time == mother_epoch_sd)[0][0]

# We use a function to compute the SD (Single (temporal) Difference) phase values
sd_complex_vals, sd_amplitude_vals, sd_phase_vals = compute_sd(stm, mother_epoch_sd)

# Safe the values to the STM
stm['sd_complex'] = xr.DataArray(sd_complex_vals, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['sd_amplitude'] = xr.DataArray(sd_amplitude_vals, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['sd_phase'] = xr.DataArray(sd_phase_vals, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})


# Loop trought the points and safe amplitude statistics for the VCM in the complex domain
sigma_ampl_sd_stm = np.zeros((len(stm.space), len(stm.time)))
mean_ampl_sd_stm = np.zeros((len(stm.space), len(stm.time)))
mad_ampl_sd_stm = np.zeros((len(stm.space), len(stm.time)))
median_ampl_sd_stm = np.zeros((len(stm.space), len(stm.time)))


for i in range(len(stm.space)):
    sigma_ampl_sd_stm[i,:], mean_ampl_sd_stm[i,:],mad_ampl_sd_stm[i,:], median_ampl_sd_stm[i,:]  = define_complex_stochastics_partition_stm(stm.isel(space = i).breakpoints_stm, stm.isel(space = i).sd_amplitude, 'normal')

# Save the values to the STM
stm['sigma_ampl_sd_stm'] = xr.DataArray(sigma_ampl_sd_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['mean_ampl_sd_stm'] = xr.DataArray(mean_ampl_sd_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['mad_ampl_sd_stm'] = xr.DataArray(mad_ampl_sd_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['median_ampl_sd_stm'] = xr.DataArray(median_ampl_sd_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})


In [18]:
# Safe the stm to a .zarr file 


#save stm to zarr file


stm_save = stm

stm_save = stm_save.drop_vars('breakpoints')
stm_save = stm_save.drop_vars('idx_outliers')




stm_save = stm_save.chunk({'space': -1, 'time': 'auto'})

# Sla de dataset op naar Zarr
# stm_save.to_zarr("output_data/stm_amsterdam_6000p_madincl.zarr")